In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"     # xet 끔 (재구성 캐시 폭증 방지)
os.environ["HF_HOME"] = "/workspace/hf"     # 큰 볼륨에 받기
!pip -q install -U transformers accelerate

In [ ]:
import json, re, torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

CHUNKS = "chunks-3dfc9d58.json"   # 파일명 다르면 여기 수정
seed = json.load(open('seed_v1.json', encoding='utf-8'))
cmap = {c['chunk_id']: c for c in json.load(open(CHUNKS, encoding='utf-8'))}

SYSTEM = ("당신은 지방계약법령에 따라 공공 SW 용역계약서 조항을 검토하는 전문가입니다. "
          "검토조항을 참고조항에 비추어 아래 형식으로만 답하십시오.\n"
          "판정: 일치|불일치\n방향: 을불리|을유리|-\n유형: A|B|-\n근거: (조항명)\n코멘트: (간단히)")
def build_user(it):
    refs="\n".join(f"[{i}] {cmap[g]['law_name']} {cmap[g]['article_id']} — {cmap[g]['text'][:250]}"
                   for i,g in enumerate([x for x in it['ref_answers'] if x in cmap],1))
    return f"카테고리: {it['category']}\n검토조항: {it['clause_text']}\n\n참고조항:\n{refs}"
def parse(t,k):
    m=re.search(rf"{k}\s*[:：]\s*(.+)",t); return m.group(1).strip() if m else ""
def jo(s): return set(re.findall(r"제\d+조(?:의\d+)?",s))

def eval_model(name):
    tok=AutoTokenizer.from_pretrained(name, trust_remote_code=True, cache_dir="/workspace/hf")
    model=AutoModelForCausalLM.from_pretrained(
        name, torch_dtype=torch.bfloat16, device_map="auto",
        trust_remote_code=True, low_cpu_mem_usage=True, cache_dir="/workspace/hf").eval()
    s={"판정":0,"방향":0,"방향N":0,"유형":0,"근거":0,"형식":0}; preds=[]
    for it in tqdm(seed, desc=name.split('/')[-1]):
        msgs=[{"role":"system","content":SYSTEM},{"role":"user","content":build_user(it)}]
        kw={"enable_thinking":False} if "qwen3" in name.lower() else {}
        text=tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, **kw)
        ids=tok(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out=model.generate(**ids, max_new_tokens=128, do_sample=False, pad_token_id=tok.eos_token_id)
        gen=tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
        pj,pd,pt,pg=parse(gen,"판정"),parse(gen,"방향"),parse(gen,"유형"),parse(gen,"근거")
        if pj: s["형식"]+=1
        if it["label"] in pj: s["판정"]+=1
        if it["direction"]:
            s["방향N"]+=1
            if it["direction"] in pd: s["방향"]+=1
            if it["type"] and it["type"] in pt: s["유형"]+=1
        goldjo=set().union(*[jo(cmap[g]["article_id"]) for g in it["ref_answers"] if g in cmap])
        if goldjo & jo(pg): s["근거"]+=1
        preds.append({"id":it["id"],"label":it["label"],"pred":gen})
    n=len(seed); dn=s["방향N"] or 1
    print(f"\n=== {name} ===\n판정 {s['판정']/n:.1%} | 방향 {s['방향']/dn:.1%} | 유형 {s['유형']/dn:.1%} | 근거 {s['근거']/n:.1%} | 형식 {s['형식']/n:.1%}")
    json.dump(preds, open(f"/workspace/pred_{name.split('/')[-1]}.json","w",encoding="utf-8"), ensure_ascii=False, indent=2)
    del model, tok; gc.collect(); torch.cuda.empty_cache()

In [ ]:
eval_model("Qwen/Qwen3-8B")
# eval_model("LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct") # 실패
eval_model("kakaocorp/kanana-1.5-8b-instruct-2505")